<a href="https://colab.research.google.com/github/marcelojr14/dashboard-vendas/blob/main/SQLAlch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sqlalchemy pandas

In [3]:
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine('sqlite:///sistema_rh.db')

In [5]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS funcionarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            cargo TEXT NOT NULL,
            salario REAL NOT NULL
        )
    """))

In [6]:
nome = "João Silva"
cargo = "Desenvolvedor Júnior"
salario = 3500.00

with engine.begin() as conn:
    conn.execute(
        text("""
            INSERT INTO funcionarios (nome, cargo, salario)
            VALUES (:nome, :cargo, :salario)
        """),
        {
            "nome": nome,
            "cargo": cargo,
            "salario": salario
        }
    )

In [7]:
df = pd.read_sql_query(
    "SELECT * FROM funcionarios",
    engine
)

df

,id,nome,cargo,salario
0,1,João Silva,Desenvolvedor Júnior,3500.0


In [8]:
from sqlalchemy import (
    Table,
    MetaData,
    Column,
    Integer,
    String
)

In [9]:
metadata = MetaData()

projetos = Table(
    'projetos',
    metadata,
    Column('id', Integer, primary_key=True, autoincrement=True),
    Column('nome', String, nullable=False),
    Column('descricao', String)
)

metadata.create_all(engine)

In [10]:
lista_de_projetos = [
    {
        "nome": "Sistema Financeiro",
        "descricao": "Desenvolvimento de sistema financeiro"
    },
    {
        "nome": "Aplicativo RH",
        "descricao": "Aplicativo para gestão de funcionários"
    },
    {
        "nome": "Portal Corporativo",
        "descricao": "Portal interno da empresa"
    }
]

In [11]:
from sqlalchemy import insert

with engine.begin() as conn:
    conn.execute(
        insert(projetos),
        lista_de_projetos
    )

In [15]:
from sqlalchemy import update


In [17]:
from sqlalchemy import Table

funcionarios = Table(
    'funcionarios',
    metadata,
    autoload_with=engine
)

In [18]:
with engine.begin() as conn:
    conn.execute(
        update(funcionarios)
        .where(funcionarios.c.cargo == 'Desenvolvedor Júnior')
        .values(salario=funcionarios.c.salario * 1.10)
    )

In [19]:
from sqlalchemy import select, func

In [20]:
consulta = (
    select(
        funcionarios.c.cargo,
        func.avg(funcionarios.c.salario).label('media_salarial')
    )
    .group_by(funcionarios.c.cargo)
)

In [21]:
with engine.connect() as conn:
    resultado = conn.execute(consulta)

    for linha in resultado:
        print(linha)

('Desenvolvedor Júnior', 3850.0000000000005)


In [22]:
from sqlalchemy.orm import (
    declarative_base,
    mapped_column,
    relationship,
    sessionmaker
)

from sqlalchemy import (
    Integer,
    String,
    ForeignKey
)

In [23]:
Base = declarative_base()

In [24]:
class Departamento(Base):
    __tablename__ = 'departamentos'

    id = mapped_column(Integer, primary_key=True)
    nome = mapped_column(String, nullable=False)

    funcionarios = relationship(
        'FuncionarioORM',
        back_populates='departamento'
    )

In [25]:
class FuncionarioORM(Base):
    __tablename__ = 'funcionarios_orm'

    id = mapped_column(Integer, primary_key=True)
    nome = mapped_column(String, nullable=False)
    cargo = mapped_column(String, nullable=False)
    salario = mapped_column(Integer, nullable=False)

    departamento_id = mapped_column(
        ForeignKey('departamentos.id')
    )

    departamento = relationship(
        'Departamento',
        back_populates='funcionarios'
    )

In [26]:
Base.metadata.create_all(engine)

In [27]:
Session = sessionmaker(bind=engine)
session = Session()

In [28]:
departamento = Departamento(
    nome="Tecnologia"
)

In [29]:
funcionario1 = FuncionarioORM(
    nome="Carlos",
    cargo="Desenvolvedor",
    salario=5000
)

funcionario2 = FuncionarioORM(
    nome="Maria",
    cargo="Analista",
    salario=4500
)

In [30]:
departamento.funcionarios.append(funcionario1)
departamento.funcionarios.append(funcionario2)

In [31]:
session.add(departamento)

session.commit()